# Accessing Claude with the API

Course: [Claude with the Anthropic API](https://anthropic.skilljar.com/claude-with-the-anthropic-api/287725)

## Setup

In [16]:
%pip install anthropic python-dotenv

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [17]:
from dotenv import load_dotenv
load_dotenv()

True

## Making a request

In [18]:
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-5"

In [29]:


def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)
    
def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    for block in message.content:
        if block.type == "text":
            return block.text

message = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=[
        {
            "role": "user",
            "content": "Write another sentence"
        }
    ]
)



In [20]:

messages = []
add_user_message(messages, "Write a 1 sentence description of a fake database")

stream = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    stream=True
)

for event in stream:
    print(event)

RawMessageStartEvent(message=Message(id='msg_011CdKahNS4DjKzcARdSZzGv', container=None, content=[], model='claude-sonnet-5', role='assistant', stop_details=None, stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=21, output_tokens=1, output_tokens_details=None, server_tool_use=None, service_tier='standard')), type='message_start')
RawContentBlockStartEvent(content_block=TextBlock(citations=None, text='', type='text'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=TextDelta(text='**', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text='UserVault** is a cloud-based NoSQL database that stores user profile information with', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelt

In [22]:
messages = []
add_user_message(messages, "Write a 1 sentence description of a fake database")

with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:
    for text in stream.text_stream:
       # print(text, end="")
       pass
stream.get_final_message()

ParsedMessage(id='msg_011CdKb91Ww4Dki5ofxj84vG', container=None, content=[ParsedTextBlock(citations=None, text='A cloud-based NoSQL database called "DataNexus" that promises infinite scalability, sub-millisecond query times, and automatic self-healing—though in reality it\'s just a glorified spreadsheet running on a single overworked server in someone\'s closet.', type='text', parsed_output=None)], model='claude-sonnet-5', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=21, output_tokens=83, output_tokens_details=None, server_tool_use=None, service_tier='standard'))

In [44]:
# Start with an empty message list
messages = []

# Add the initial user question
add_user_message(messages, "Define quantum computing in one sentence")
 
# Get Claude's response
answer = chat(messages)

# Add Claude's response to the conversation history
add_assistant_message(messages, answer)
 
# Add a follow-up question
add_user_message(messages, "Write another sentence") 

# Get the follow-up response with full context
answer = chat(messages)
answer

'**Quantum computing** harnesses qubits, which unlike classical bits can exist in multiple states simultaneously, enabling massively parallel calculations for tasks like cryptography, optimization, and molecular simulation.'

In [ ]:
messages = []

while True:
    user_input = input("> ")
    print(">", user_input)
    add_user_message(messages, user_input)
    answer = chat(messages)
    add_assistant_message(messages, answer)
    print("----")
    print(answer)
    print("----")
    

> whats 1+1?
----
**1 + 1 = 2**

A basic arithmetic fact! Is there something more complex I can help you calculate?
----
> and 2 more?
----
**2 + 2 = 4**

Let me know if you'd like to keep going or need help with anything else!
----


In [18]:
messages = []
system_prompt = """
    You are a patient math tutor.
    Do not directly answer a student's questions.
    Guide them to a solution step by step.
    """         
add_user_message(messages, "how do I solve 5x+3=2 for x?")
answer = chat(messages, system=system_prompt)
answer

'I\'d be happy to write another sentence, but this appears to be the start of our conversation, so I don\'t have a previous sentence to follow up on. Could you let me know:\n\n1. What topic you\'d like the sentence to be about, or\n2. Share the original sentence you\'d like me to follow up on?\n\nFor example, I could write something like: "The sun set slowly behind the mountains, painting the sky in brilliant shades of orange and pink."\n\nLet me know what you had in mind!'

In [14]:
messages=[]
add_user_message(messages, "generate a one sentece move idea")

answer = chat(messages, system="you are a movie director", temperature=1.0)
answer

"A retired assassin discovers her final target from decades ago has been living next door, raising her grandchildren, and now she must decide whether to finish the job or protect the family she's grown to love."

In [30]:
messages = []
add_user_message(messages, "Generate a very short event bridge rule as json")   
add_assistant_message(messages, "```json")
chat(messages,stop_sequences=["```"])

BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'This model does not support assistant message prefill. The conversation must end with a user message.'}, 'request_id': 'req_011CdKdZmL61XP9Zb4UpREXX'}